In [ ]:
import os

print(os.listdir("/kaggle/input"))

In [ ]:
import os

dataset_path = "/kaggle/input/datasets/raheem1405/ucf-subset-v1/ucf_subset_v1"

normal_path = os.path.join(dataset_path, "normal")
anomaly_path = os.path.join(dataset_path, "anomaly")

video_paths = []
labels = []

# Normal videos → label 0
for vid in os.listdir(normal_path):
    if vid.endswith(".mp4"):
        video_paths.append(os.path.join(normal_path, vid))
        labels.append(0)

# Anomaly videos → label 1
for vid in os.listdir(anomaly_path):
    if vid.endswith(".mp4"):
        video_paths.append(os.path.join(anomaly_path, vid))
        labels.append(1)

print("Total videos:", len(video_paths))

In [ ]:
import pandas as pd

df = pd.DataFrame({
    "video_path": video_paths,
    "label": labels
})

df.to_csv("/kaggle/working/labels.csv", index=False)

In [ ]:
import pandas as pd

df = pd.read_csv("/kaggle/input/YOUR-DATASET/labels.csv")

video_paths = df["video_path"].tolist()
labels = df["label"].tolist()

In [ ]:
from sklearn.model_selection import train_test_split

# Step 1: shuffle + split into train and temp
X_train, X_temp, y_train, y_temp = train_test_split(
    video_paths, labels, test_size=0.3, shuffle=True, random_state=42
)

# Step 2: split temp into val + testZ
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, shuffle=True, random_state=42
)

In [ ]:
import cv2
import numpy as np
import random

def batch_generator(video_paths, labels, batch_size=8, clip_length=30, frame_size=(128,128)):
    
    video_paths = np.array(video_paths)
    labels = np.array(labels)
    
    while True:
        
        indices = np.arange(len(video_paths))
        np.random.shuffle(indices)
        
        X_batch = []
        y_batch = []
        
        for idx in indices:
            
            path = video_paths[idx]
            label = labels[idx]
            
            cap = cv2.VideoCapture(path)
            frames = []
            
            # Extract frames
            while True:
                ret, frame = cap.read()
                if not ret:
                    break
                
                frame = cv2.resize(frame, frame_size)
                frame = frame.astype(np.float32) / 255.0
                frames.append(frame)
            
            cap.release()
            
            # Skip short videos
            if len(frames) < clip_length:
                continue
            
            # 🔥 RANDOM CLIP FROM VIDEO (KEY FIX)
            start = random.randint(0, len(frames) - clip_length)
            clip = frames[start:start + clip_length]
            
            X_batch.append(clip)
            y_batch.append(label)
            
            # Yield batch
            if len(X_batch) == batch_size:
                yield np.array(X_batch), np.array(y_batch)
                
                X_batch = []
                y_batch = []

In [ ]:
X, y = next(batch_generator(X_train, y_train, batch_size=4))
print(y)

In [ ]:
gen = batch_generator(video_paths, labels, batch_size=4)

X_batch, y_batch = next(gen)

print("X shape:", X_batch.shape)
print("y:", y_batch)

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten
from tensorflow.keras.layers import Dense, Dropout, LSTM, TimeDistributed

In [ ]:
print("GPU:", tf.config.list_physical_devices('GPU'))

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten
from tensorflow.keras.layers import Dense, LSTM, TimeDistributed, Dropout
from tensorflow.keras.layers import GlobalAveragePooling2D
from tensorflow.keras import Input

model = Sequential()

model.add(Input(shape=(30, 128, 128, 3))) 

# ----- CNN (applied to each frame) -----
model.add(TimeDistributed(Conv2D(32, (3,3), activation='relu'),
                          input_shape=(30, 128, 128, 3)))
model.add(TimeDistributed(MaxPooling2D(2,2)))

model.add(TimeDistributed(Conv2D(64, (3,3), activation='relu')))
model.add(TimeDistributed(MaxPooling2D(2,2)))

model.add(TimeDistributed(Conv2D(128, (3,3), activation='relu')))
model.add(TimeDistributed(MaxPooling2D(2,2)))

model.add(TimeDistributed(GlobalAveragePooling2D()))

# ----- LSTM (temporal learning) -----
model.add(LSTM(128))

# ----- Dense (classification) -----
model.add(Dense(64, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(1, activation='sigmoid'))   # binary classification

In [ ]:
model.compile(
    # optimizer='adam',
    optimizer = tf.keras.optimizers.Adam(learning_rate=0.0005)
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [ ]:
import numpy as np
print(np.bincount(labels))

In [ ]:
def val_generator(video_paths, labels, batch_size=4, clip_length=30, frame_size=(128,128)):
    
    while True:
        X_batch = []
        y_batch = []
        
        for path, label in zip(video_paths, labels):
            
            cap = cv2.VideoCapture(path)
            frames = []
            
            while True:
                ret, frame = cap.read()
                if not ret:
                    break
                
                frame = cv2.resize(frame, frame_size)
                frame = frame.astype(np.float32) / 255.0
                frames.append(frame)
            
            cap.release()
            
            if len(frames) < clip_length:
                continue
            
            # 🔥 FIXED CLIP (NO RANDOM)
            clip = frames[0:clip_length]
            
            X_batch.append(clip)
            y_batch.append(label)
            
            if len(X_batch) == batch_size:
                yield np.array(X_batch), np.array(y_batch)
                
                X_batch = []
                y_batch = []

In [ ]:
batch_size = 4

model.fit(
    batch_generator(X_train, y_train, batch_size),   # ✅ TRAIN (random)

    validation_data=val_generator(X_val, y_val, batch_size),  # ✅ VAL (fixed)

    steps_per_epoch=len(X_train)//batch_size,
    validation_steps=len(X_val)//batch_size,

    epochs=20
)

In [ ]:
model.save("/kaggle/working/anomaly_model_v1.keras")

In [ ]:
X, y = next(batch_generator(X_train, y_train, 4))
preds = model.predict(X)

print(preds)

In [ ]:
import numpy as np
print("Validation labels:", np.bincount(y_val))

Training with the second dataset : 

In [ ]:
import os

dataset_path = "/kaggle/input/datasets/raheem1405/ucf-subset-v2/ucf_subset_v1"

normal_path = os.path.join(dataset_path, "normal")
anomaly_path = os.path.join(dataset_path, "anomaly")

video_paths = []
labels = []

# Normal videos → label 0
for vid in os.listdir(normal_path):
    if vid.endswith(".mp4"):
        video_paths.append(os.path.join(normal_path, vid))
        labels.append(0)

# Anomaly videos → label 1
for vid in os.listdir(anomaly_path):
    if vid.endswith(".mp4"):
        video_paths.append(os.path.join(anomaly_path, vid))
        labels.append(1)

print("Total videos:", len(video_paths))

In [ ]:
import pandas as pd

df = pd.DataFrame({
    "video_path": video_paths,
    "label": labels
})

df.to_csv("/kaggle/working/labels2.csv", index=False)

In [ ]:
import os
print(os.listdir("/kaggle/working"))

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Load the expanded dataset
df = pd.read_csv("/kaggle/working/labels2.csv")

# Separate features and labels
X = df['video_path'].tolist()
y = df['label'].tolist()

# Step 1: shuffle + split into train and temp (70/30)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, shuffle=True, stratify=y, random_state=42
)

# Step 2: split temp into val + test (50/50 of temp → 15% val, 15% test)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, shuffle=True, stratify=y_temp, random_state=42
)

print("Train:", len(X_train), "Validation:", len(X_val), "Test:", len(X_test))

In [ ]:
import cv2
import numpy as np
import random

def batch_generator_df(df, batch_size=4, clip_length=20, frame_size=(128,128)):
    """
    Batch generator for videos from a DataFrame containing video paths and labels.
    
    Args:
        df: pandas DataFrame with columns ['video_path', 'label']
        batch_size: number of clips per batch
        clip_length: number of frames per clip
        frame_size: resize each frame to (H, W)
        
    Yields:
        X_batch: np.array of shape (batch_size, clip_length, H, W, 3)
        y_batch: np.array of shape (batch_size,)
    """
    
    video_paths = df['video_path'].tolist()
    labels = df['label'].tolist()
    
    video_paths = np.array(video_paths)
    labels = np.array(labels)
    
    while True:
        indices = np.arange(len(video_paths))
        np.random.shuffle(indices)
        
        X_batch = []
        y_batch = []
        
        for idx in indices:
            path = video_paths[idx]
            label = labels[idx]
            
            cap = cv2.VideoCapture(path)
            frames = []
            
            while True:
                ret, frame = cap.read()
                if not ret:
                    break
                frame = cv2.resize(frame, frame_size)
                frame = frame.astype(np.float32) / 255.0
                frames.append(frame)
            
            cap.release()
            
            if len(frames) < clip_length:
                continue
            
            start = random.randint(0, len(frames) - clip_length)
            clip = frames[start:start + clip_length]
            
            X_batch.append(clip)
            y_batch.append(label)
            
            if len(X_batch) == batch_size:
                yield np.array(X_batch), np.array(y_batch)
                X_batch = []
                y_batch = []

In [ ]:
train_gen = batch_generator_df(pd.DataFrame({'video_path': X_train, 'label': y_train}), batch_size=4, clip_length=20)
val_gen   = batch_generator_df(pd.DataFrame({'video_path': X_val, 'label': y_val}), batch_size=4, clip_length=20)
test_gen  = batch_generator_df(pd.DataFrame({'video_path': X_test, 'label': y_test}), batch_size=4, clip_length=20)

In [ ]:
X_sample, y_sample = next(train_gen)
print(X_sample.shape)

In [ ]:
from tensorflow.keras.models import load_model

model = load_model("/kaggle/working/anomaly_model_v1.keras")
print("Model loaded successfully!")

In [ ]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [ ]:
batch_size = 4

steps_per_epoch = len(X_train) // batch_size
validation_steps = len(X_val) // batch_size

print("Steps:", steps_per_epoch, "Val steps:", validation_steps)

In [ ]:
# history = model.fit(
#     train_gen,
#     steps_per_epoch=steps_per_epoch,
#     validation_data=val_gen,
#     validation_steps=validation_steps,
#     epochs=10,
#     # class_weight={0: 2.0, 1: 1.0}
# )
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

# 🔥 Save BEST model automatically
checkpoint = ModelCheckpoint(
    "/kaggle/working/anomaly_model_v2.keras",
    monitor='val_loss',
    save_best_only=True,
    verbose=1
)

# 🔥 Stop early when model starts overfitting
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

# 🚀 Training
history = model.fit(
    train_gen,
    steps_per_epoch=steps_per_epoch,
    validation_data=val_gen,
    validation_steps=validation_steps,
    epochs=10,
    callbacks=[checkpoint, early_stop]   # ✅ ADDED THIS
)

In [ ]:
import tensorflow as tf
tf.keras.backend.clear_session()

In [ ]:
import os

print(os.listdir("/kaggle/working"))

In [ ]:
from tensorflow.keras.models import load_model

model = load_model("/kaggle/working/anomaly_model_v2.keras")

In [ ]:
model.evaluate(test_gen, steps=len(X_test)//4)

In [ ]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
import keras

layer = keras.layers.TFSMLayer(
    "/kaggle/input/datasets/raheem1405/pre-trained-model/Model_Trained/saved_model/my_model",
    call_endpoint="serving_default"
)

In [ ]:
from keras import Input, Model

inputs = Input(shape=(20, 128, 128, 3))  # same as your clips
outputs = layer(inputs)

model = Model(inputs, outputs)

In [ ]:
import os

dataset_path = "/kaggle/input/datasets/raheem1405/ucf-subset-v2/ucf_subset_v1"

normal_path = os.path.join(dataset_path, "normal")
anomaly_path = os.path.join(dataset_path, "anomaly")

video_paths = []
labels = []

# Normal videos → label 0
for vid in os.listdir(normal_path):
    if vid.endswith(".mp4"):
        video_paths.append(os.path.join(normal_path, vid))
        labels.append(0)

# Anomaly videos → label 1
for vid in os.listdir(anomaly_path):
    if vid.endswith(".mp4"):
        video_paths.append(os.path.join(anomaly_path, vid))
        labels.append(1)

print("Total videos:", len(video_paths))

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Load the expanded dataset
df = pd.read_csv("/kaggle/working/labels2.csv")

# Separate features and labels
X = df['video_path'].tolist()
y = df['label'].tolist()

# Step 1: shuffle + split into train and temp (70/30)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, shuffle=True, stratify=y, random_state=42
)

# Step 2: split temp into val + test (50/50 of temp → 15% val, 15% test)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, shuffle=True, stratify=y_temp, random_state=42
)

print("Train:", len(X_train), "Validation:", len(X_val), "Test:", len(X_test))

In [ ]:
# Create train/val/test DataFrames for generator
train_df = pd.DataFrame({'video_path': X_train, 'label': y_train})
val_df   = pd.DataFrame({'video_path': X_val, 'label': y_val})
test_df  = pd.DataFrame({'video_path': X_test, 'label': y_test})

print("Train DF:", len(train_df), "Val DF:", len(val_df), "Test DF:", len(test_df))

In [ ]:
from tensorflow.keras.applications.resnet50 import preprocess_input
import numpy as np
import cv2
import random

def batch_generator_df_resnet(df, batch_size=4, clip_length=20, frame_size=(224,224)):
    video_paths = df['video_path'].tolist()
    labels = df['label'].tolist()
    video_paths = np.array(video_paths)
    labels = np.array(labels)
    
    while True:
        indices = np.arange(len(video_paths))
        np.random.shuffle(indices)
        
        X_batch = []
        y_batch = []
        
        for idx in indices:
            path = video_paths[idx]
            label = labels[idx]
            
            cap = cv2.VideoCapture(path)
            frames = []
            
            while True:
                ret, frame = cap.read()
                if not ret or len(frames) >= clip_length + 5:
                    break
                frame = cv2.resize(frame, frame_size)
                frame = frame.astype(np.float32)
                frame = preprocess_input(frame)  # ResNet preprocessing
                frames.append(frame)
            
            cap.release()
            
            # PAD short clips instead of skipping
            if len(frames) == 0:
                continue  # skip completely empty videos
            while len(frames) < clip_length:
                frames.append(frames[-1])  # pad with last frame
            
            # Random clip selection
            start = random.randint(0, len(frames) - clip_length)
            clip = frames[start:start + clip_length]
            
            X_batch.append(clip)
            y_batch.append(label)
            
            if len(X_batch) == batch_size:
                yield np.array(X_batch), np.array(y_batch)
                X_batch = []
                y_batch = []

In [ ]:
train_gen = batch_generator_df_resnet(train_df, batch_size=2, clip_length=20)
val_gen   = batch_generator_df_resnet(val_df, batch_size=2, clip_length=20)
test_gen  = batch_generator_df_resnet(test_df, batch_size=2, clip_length=20)

In [ ]:
# batch_size = 4
steps_per_epoch = len(train_df) // 4
validation_steps = len(val_df) // 4
test_steps = len(test_df) // 4

print("Steps per epoch:", steps_per_epoch, 
      "Validation steps:", validation_steps, 
      "Test steps:", test_steps)

In [ ]:
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, TimeDistributed, GlobalAveragePooling2D, Attention

num_classes = 2  # binary classification: normal vs anomaly

inputs = Input(shape=(20, 224, 224, 3))

# ResNet backbone (pretrained)
resnet_base = ResNet50(weights='imagenet', include_top=False)
for layer in resnet_base.layers:
    layer.trainable = False  # Phase 1: freeze ResNet

# TimeDistributed ResNet + GAP
x = TimeDistributed(resnet_base)(inputs)
x = TimeDistributed(GlobalAveragePooling2D())(x)

# LSTM + Attention
x = LSTM(128, return_sequences=True)(x)
attention = Attention()([x, x])
x = Dense(64, activation='relu')(attention)
outputs = Dense(num_classes, activation='softmax')(x[:, -1, :])  # last timestep output

model = Model(inputs, outputs)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

In [ ]:
import numpy as np
import cv2
import random

In [ ]:
history = model.fit(
    train_gen,
    steps_per_epoch=steps_per_epoch,
    validation_data=val_gen,
    validation_steps=validation_steps,
    epochs=5  # quick test run; increase after sanity check
)

In [ ]:
# Save as H5
model.save("/kaggle/working/v3_binary_resnet_lstm.h5")

# Save as Keras format
model.save("/kaggle/working/v3_binary_resnet_lstm.keras")

In [ ]:
# Unfreeze last ResNet layers for fine-tuning
for layer in model.layers[1].layer.layers:  # model.layers[1] = TimeDistributed(ResNet)
    layer.trainable = False  # freeze all first
for layer in model.layers[1].layer.layers[-30:]:  # unfreeze last 30 layers
    layer.trainable = True

In [ ]:
from tensorflow.keras.optimizers import Adam

model.compile(
    optimizer=Adam(learning_rate=1e-5),  # low LR for fine-tuning
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
steps_per_epoch = len(train_df) // 4  # batch_size = 4
validation_steps = len(val_df) // 4

In [ ]:
history = model.fit(
    train_gen,
    steps_per_epoch=steps_per_epoch,
    validation_data=val_gen,
    validation_steps=validation_steps,
    epochs=20,  # crank up
    verbose=1
)

In [ ]:
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

# 🔹 Step 1: Lower learning rate for smooth fine-tuning
model.compile(
    optimizer=Adam(learning_rate=5e-6),  # strict low LR
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# 🔹 Step 2: Callbacks for optimal training
checkpoint = ModelCheckpoint(
    '/kaggle/working/v3_binary_resnet_lstm_best.keras',
    monitor='val_accuracy',
    save_best_only=True,
    mode='max',
    verbose=1
)

early_stop = EarlyStopping(
    monitor='val_accuracy',
    patience=5,          # stop if no improvement for 5 epochs
    verbose=1,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,          # halve LR if loss plateaus
    patience=3,
    verbose=1,
    min_lr=1e-7
)

# 🔹 Step 3: Fine-tune
history = model.fit(
    train_gen,
    steps_per_epoch=steps_per_epoch,
    validation_data=val_gen,
    validation_steps=validation_steps,
    epochs=30,            # max epochs, callbacks handle early stop
    callbacks=[checkpoint, early_stop, reduce_lr],
    verbose=1
)

In [ ]:
import os
print(os.listdir('/kaggle/working'))

In [ ]:
from tensorflow.keras.models import load_model
from tensorflow.keras.layers import Layer

# Dummy GetItem layer fix
class GetItem(Layer):
    def call(self, inputs):
        return inputs[:, -1, :]

model = load_model(
    '/kaggle/working/v3_binary_resnet_lstm_best.h5',
    custom_objects={'GetItem': GetItem}
)

In [ ]:
from tensorflow.keras.models import load_model

model = load_model('/kaggle/working/v3_binary_resnet_lstm.keras')

In [ ]:
from tensorflow.keras.models import load_model

model_v3 = load_model('/kaggle/working/v3_binary_resnet_lstm_best.keras')
model_base = load_model('/kaggle/working/v3_binary_resnet_lstm.keras')

In [ ]:
test_steps = len(test_df) // 2   # batch_size = 2

In [ ]:
print("V3 BEST MODEL:")
model_v3.evaluate(test_gen, steps=test_steps)

print("\nBASE RESNET MODEL:")
model_base.evaluate(test_gen, steps=test_steps)

In [ ]:
from tensorflow.keras.applications.resnet50 import preprocess_input
import numpy as np
import cv2

def test_generator_fixed(df, batch_size=2, clip_length=20, frame_size=(224,224)):
    video_paths = df['video_path'].tolist()
    labels = df['label'].tolist()
    
    video_paths = np.array(video_paths)
    labels = np.array(labels)
    
    while True:
        indices = np.arange(len(video_paths))  # ✅ NO SHUFFLE
        
        X_batch = []
        y_batch = []
        
        for idx in indices:
            path = video_paths[idx]
            label = labels[idx]
            
            cap = cv2.VideoCapture(path)
            frames = []
            
            while True:
                ret, frame = cap.read()
                if not ret or len(frames) >= clip_length:
                    break
                frame = cv2.resize(frame, frame_size)
                frame = frame.astype(np.float32)
                frame = preprocess_input(frame)
                frames.append(frame)
            
            cap.release()
            
            if len(frames) == 0:
                continue
            
            # PAD if needed
            while len(frames) < clip_length:
                frames.append(frames[-1])
            
            # ✅ NO RANDOM START — deterministic
            clip = frames[:clip_length]
            
            X_batch.append(clip)
            y_batch.append(label)
            
            if len(X_batch) == batch_size:
                yield np.array(X_batch), np.array(y_batch)
                X_batch = []
                y_batch = []

In [ ]:
test_gen_fixed = test_generator_fixed(test_df, batch_size=2, clip_length=20)

In [ ]:
pred_v3 = model_v3.predict(test_gen_fixed, steps=test_steps)

# IMPORTANT: recreate generator again (so it starts fresh)
test_gen_fixed = test_generator_fixed(test_df, batch_size=2, clip_length=20)

pred_base = model_base.predict(test_gen_fixed, steps=test_steps)

In [ ]:
import numpy as np

final_pred = (pred_v3 + pred_base) / 2
y_pred = np.argmax(final_pred, axis=1)

In [ ]:
y_true = test_df['label'].values[:len(y_pred)]

In [ ]:
from sklearn.metrics import accuracy_score

ensemble_acc = accuracy_score(y_true, y_pred)
print("🔥 Ensemble Accuracy:", ensemble_acc)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

In [ ]:
from tensorflow.keras.models import load_model

model = load_model("/kaggle/input/models/raheem1405/best/keras/default/1/v3_binary_resnet_lstm_best.keras")
print("Model loaded ✅")

In [ ]:
import cv2
import numpy as np

IMG_SIZE = 224
SEQUENCE_LENGTH = 20

def preprocess_video(video_path):
    cap = cv2.VideoCapture(video_path)
    frames = []

    while len(frames) < SEQUENCE_LENGTH:
        ret, frame = cap.read()
        if not ret:
            break
        
        frame = cv2.resize(frame, (IMG_SIZE, IMG_SIZE))
        frame = frame / 255.0
        frames.append(frame)

    cap.release()

    while len(frames) < SEQUENCE_LENGTH:
        frames.append(np.zeros((IMG_SIZE, IMG_SIZE, 3)))

    return np.array(frames)

In [ ]:
X_test = []
y_test = []

# NORMAL videos (label = 0)
X_test.append(preprocess_video("/kaggle/input/yourdata/normal1.mp4"))
y_test.append(0)

# ANOMALY videos (label = 1)
X_test.append(preprocess_video("/kaggle/input/yourdata/anomaly1.mp4"))
y_test.append(1)

X_test = np.array(X_test)
y_test = np.array(y_test)

In [ ]:
from sklearn.metrics import f1_score
import numpy as np

y_pred_probs = model.predict(X_test)

# FIX
y_pred = np.argmax(y_pred_probs, axis=1)

print("y_test:", y_test)
print("y_pred:", y_pred)

f1 = f1_score(y_test, y_pred)
print("F1 Score:", f1)

In [ ]:
print("y_test shape:", y_test.shape)
print("y_pred_probs shape:", y_pred_probs.shape)

final implementation

In [ ]:
from tensorflow.keras.models import load_model

model = load_model("/kaggle/input/models/raheem1405/best/keras/default/1/v3_binary_resnet_lstm_best.keras")
print("Model loaded ✅")

In [ ]:
IMG_SIZE = 224
SEQUENCE_LENGTH = 20

In [ ]:
import cv2
import numpy as np
from tensorflow.keras.models import load_model

model = load_model("/kaggle/input/models/raheem1405/best/keras/default/1/v3_binary_resnet_lstm_best.keras")

IMG_SIZE = 224
SEQUENCE_LENGTH = 20

video_path = "/kaggle/input/datasets/raheem1405/ucf-subset-v2/ucf_subset_v1/anomaly/abuse_02.mp4"
cap = cv2.VideoCapture(video_path)

# Get video properties
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

# Output video
fourcc = cv2.VideoWriter_fourcc(*'XVID')
out = cv2.VideoWriter('output.avi', fourcc, 20.0, (width, height))

frames = []

while True:
    ret, frame = cap.read()
    if not ret:
        break

    display_frame = frame.copy()

    # preprocess
    frame_resized = cv2.resize(frame, (IMG_SIZE, IMG_SIZE))
    frame_norm = frame_resized / 255.0
    frames.append(frame_norm)

    if len(frames) == SEQUENCE_LENGTH:
        input_seq = np.expand_dims(frames, axis=0)

        pred = model.predict(input_seq)[0]
        label = np.argmax(pred)

        if label == 1:
            text = "ANOMALY DETECTED"
            color = (0, 0, 255)
        else:
            text = "NORMAL"
            color = (0, 255, 0)

        cv2.putText(display_frame, text, (30, 50),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, color, 2)

        frames.pop(0)

    out.write(display_frame)

cap.release()
out.release()

print("✅ Output video saved as output.avi")

In [ ]:
import cv2
import numpy as np
from tensorflow.keras.models import load_model

model = load_model("/kaggle/input/models/raheem1405/best/keras/default/1/v3_binary_resnet_lstm_best.keras")

IMG_SIZE = 224
SEQUENCE_LENGTH = 20

video_path = "/kaggle/input/datasets/raheem1405/ucf-subset-v1/ucf_subset_v1/normal/normal_11.mp4"
cap = cv2.VideoCapture(video_path)

# Get video properties
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

# Output video
fourcc = cv2.VideoWriter_fourcc(*'XVID')
out = cv2.VideoWriter('outputnormal.avi', fourcc, 20.0, (width, height))

frames = []

while True:
    ret, frame = cap.read()
    if not ret:
        break

    display_frame = frame.copy()

    # preprocess
    frame_resized = cv2.resize(frame, (IMG_SIZE, IMG_SIZE))
    frame_norm = frame_resized / 255.0
    frames.append(frame_norm)

    if len(frames) == SEQUENCE_LENGTH:
        input_seq = np.expand_dims(frames, axis=0)

        pred = model.predict(input_seq)[0]
        label = np.argmax(pred)

        if label == 1:
            text = "ANOMALY DETECTED"
            color = (0, 0, 255)
        else:
            text = "NORMAL"
            color = (0, 255, 0)

        cv2.putText(display_frame, text, (30, 50),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, color, 2)

        frames.pop(0)

    out.write(display_frame)

cap.release()
out.release()

print("✅ Output video saved as output.avi")